## Experience Engine (Jupyter)

Notebook counterpart to `main.py`: same agent (local **Qwen 0.6B**), tools, and session behavior. **Verbose agent traces** and replies scroll in the left pane; the **environment view** (game or car) updates on the right.

**Run order:** (1) UI setup cell → (2) load agent → (3) attach handlers.

Commands: `exit` / `quit` / `q` (closing sequence), `EXIT` (abort, no closing), `clear` (reset chat history).

Display routing lives in `gui_viewer.py`: **`terminal`** = OpenCV window (CLI), **`jupyter`** = widget callback. This notebook calls `set_display_mode("jupyter")` before the environment starts. You can also set `DISPLAY_MODE=jupyter` in `select_environment.config`.

In [ ]:
from pathlib import Path

from IPython.display import display, HTML
import ipywidgets as widgets
import cv2
import numpy as np
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "agent.py").exists():
    display(
        HTML(
            "<p><b>Error:</b> Jupyter working directory must be the experience-engine repo "
            "(the folder that contains <code>agent.py</code>).</p>"
        )
    )
    raise RuntimeError("Wrong working directory")

load_dotenv(ROOT / ".env")

from gui_viewer import set_display_mode, set_jupyter_image_handler

set_display_mode("jupyter")

game_img = widgets.Image(
    format="png",
    layout=widgets.Layout(
        width="420px",
        max_width="48%",
        border="1px solid #ccc",
        object_fit="contain",
    ),
)


def _push_env_frame(bgr: np.ndarray) -> None:
    ok, buf = cv2.imencode(".png", bgr)
    if ok:
        game_img.value = buf.tobytes()


set_jupyter_image_handler(_push_env_frame)

log_out = widgets.Output(
    layout=widgets.Layout(
        height="640px",
        overflow_y="scroll",
        flex="1",
        min_width="340px",
        border="1px solid #ddd",
        padding="8px",
    )
)

header = widgets.HTML(
    "<h3>Experience Engine</h3>"
    "<p>Left: model + tool traces. Right: live environment frame. Use <b>Send</b> or Enter.</p>"
)

user_in = widgets.Text(placeholder="You: ", layout=widgets.Layout(width="100%"))
send_btn = widgets.Button(description="Send", button_style="primary")

row = widgets.HBox(
    [log_out, game_img],
    layout=widgets.Layout(width="100%", align_items="flex-start", gap="14px"),
)
ui = widgets.VBox([header, row, user_in, send_btn])
display(ui)

In [ ]:
from agent import create_conversational_agent
import main as main_mod
from tools.session_tools import get_session_control_signal, reset_session_control_signal
from tools import close_env

inject_image = main_mod.inject_image
session_id = "default_session"

with log_out:
    print("Initializing agent (Qwen + vector stores + environment)...")

agent, message_history = create_conversational_agent()

with log_out:
    print("Ready. Type a message and click Send (or press Enter).\n")

In [ ]:
def start_fresh_session() -> None:
    global agent, message_history
    agent, message_history = create_conversational_agent()
    with log_out:
        print("\n[Session restarted]\n")


def on_send(_=None) -> None:
    global agent, message_history
    text = user_in.value.strip()
    user_in.value = ""
    if not text:
        return

    with log_out:
        print(f"You: {text}\n")

    if text == "EXIT":
        with log_out:
            print("\n[Aborting session immediately without closing sequence]")
        close_env()
        send_btn.disabled = True
        user_in.disabled = True
        return

    if text.lower() == "clear":
        message_history.clear()
        with log_out:
            print("\n[Conversation history cleared]\n")
        return

    if text.lower() in ("exit", "quit", "q"):
        with log_out:
            main_mod.run_closing_sequence(agent, session_id, "User ended the session")
            print("Goodbye!")
        send_btn.disabled = True
        user_in.disabled = True
        return

    with log_out:
        response = agent.invoke(
            {"input": inject_image(text)},
            config={"configurable": {"session_id": session_id}},
        )
        print(f"\nAssistant: {response['output']}\n")
        print("-" * 60 + "\n")

    signal = get_session_control_signal()
    if signal["action"] == "end":
        reset_session_control_signal()
        with log_out:
            main_mod.run_closing_sequence(agent, session_id, "Agent ended the session")
            print("Goodbye!")
        send_btn.disabled = True
        user_in.disabled = True
    elif signal["action"] == "restart":
        reset_session_control_signal()
        with log_out:
            main_mod.run_closing_sequence(agent, session_id, "Agent restarting the session")
        start_fresh_session()


send_btn.on_click(on_send)
user_in.on_submit(lambda _: on_send())